# 6.2 许可证与使用限制 / Licenses and usage restrictions

能下载不等于能随意使用。几类常见的模型许可证各自允许什么、要求什么，以及怎样查到一个模型的许可证。

Being able to download a model does not mean being free to use it in any way. What the common kinds of model licenses allow and require, and how to find the license of a model.

讲解与推导见网站对应模块。每个练习先写代码，再运行紧随其后的检查单元。

The explanation and derivations are on the website. For each exercise, write the code, then run the check cell right after it.

## 1. 从 Hub 读取并分类许可证 / Read and classify licenses from the Hub

实现一个把许可证标识符归入六个粗略类别的函数，再用 `huggingface_hub` 读取模型卡元数据中的许可证信息。测试先在固定的输入上检查分类，再读取三个较早发布的模型的真实元数据，随后对 19 个模型做一次调查并打印结果。打印的表格反映的是运行当时的状态，可能与本页表格中记录的不同，这正是需要每次都查的原因。

Implement a function that sorts license identifiers into six coarse categories, and read the license information in model-card metadata with `huggingface_hub`. The tests first check the classification on fixed inputs, then read the real metadata of three older models, and then survey 19 models and print the result. The printed table reflects the state at the time of the run and may differ from the table recorded on this page, which is exactly why it has to be checked every time.

需要联网下载模型。/ Downloads a model.

In [ ]:
from huggingface_hub import HfApi

# The license of a model on the Hugging Face Hub is declared in the metadata of its model card:
#   license: apache-2.0                       an identifier from the Hub's list
#   license: other, license_name: mrl         a custom license, whose text is in the repository
# This exercise reads that metadata and sorts it into a few coarse categories. The category is a first
# orientation, not a legal judgement: the binding text is always the license itself.

PERMISSIVE = {"apache-2.0", "mit", "bsd-2-clause", "bsd-3-clause", "cc0-1.0", "cc-by-4.0", "isc", "unlicense"}
COPYLEFT = {"gpl-2.0", "gpl-3.0", "agpl-3.0", "lgpl-3.0", "mpl-2.0", "cc-by-sa-4.0"}


def classify_license(identifier, name=None):
    """Sort a license into one of six categories. Check in this order:

      "unknown"         identifier is None, empty or "unknown"
      "permissive"      identifier in PERMISSIVE
      "copyleft"        identifier in COPYLEFT
      "non-commercial"  identifier contains "-nc", "noncommercial" or "research",
                        or the name contains "research", or the name is "mrl" (Mistral Research License)
      "use-restricted"  identifier contains "rail" (the OpenRAIL family)
      "custom"          everything else: family licenses such as llama3.3 or gemma, and "other"
    Compare in lower case. `name` may be None.
    """
    # TODO 1
    raise NotImplementedError


def read_license(repo_id, api=None):
    """Read the license metadata of one model repository.

    info = (api or HfApi()).model_info(repo_id)
    info.card_data.to_dict() holds the keys "license" and "license_name" (either may be missing,
    and card_data itself may be None). info.gated is False, "auto" or "manual".
    Returns {"repo", "license", "name", "gated" (a bool), "category"}.
    """
    # TODO 2
    raise NotImplementedError


def survey(repo_ids):
    """read_license for every repository, as a list of dicts."""
    # TODO 3
    raise NotImplementedError

In [ ]:
# 检查 / checks
# 1. Classification on fixed inputs
cases = [("apache-2.0", None, "permissive"), ("MIT", None, "permissive"), ("cc-by-4.0", None, "permissive"),
         ("gpl-3.0", None, "copyleft"), ("cc-by-nc-4.0", None, "non-commercial"), ("cc-by-nc-sa-4.0", None, "non-commercial"),
         ("fair-noncommercial-research-license", None, "non-commercial"), ("other", "mrl", "non-commercial"),
         ("other", "some-research-license", "non-commercial"),
         ("openrail", None, "use-restricted"), ("bigscience-bloom-rail-1.0", None, "use-restricted"),
         ("llama3.3", None, "custom"), ("gemma", None, "custom"), ("other", "llama4", "custom"), ("other", None, "custom"),
         (None, None, "unknown"), ("", None, "unknown"), ("unknown", None, "unknown")]
for ident, name, expected in cases:
    got = classify_license(ident, name)
    assert got == expected, f"classify_license({ident!r}, {name!r}) should be {expected!r}, got {got!r}"

# 2. Live metadata from the Hub. These repositories are old and their licenses are unlikely to change.
row = read_license("openai-community/gpt2")
assert row == {"repo": "openai-community/gpt2", "license": "mit", "name": None, "gated": False, "category": "permissive"}, row
row = read_license("meta-llama/Llama-3.3-70B-Instruct")       # metadata of gated repositories is public
assert row["license"] == "llama3.3" and row["category"] == "custom" and row["gated"] is True, row
row = read_license("mistralai/Mistral-Large-Instruct-2411")
assert row["license"] == "other" and row["name"] == "mrl" and row["category"] == "non-commercial", row

# 3. A survey of well-known checkpoints. Printed, not asserted: this is exactly the information that changes.
REPOS = ["openai-community/gpt2", "HuggingFaceTB/SmolLM2-135M", "openai/gpt-oss-20b", "microsoft/phi-4",
         "deepseek-ai/DeepSeek-V4.1-Flash", "deepseek-ai/DeepSeek-V3", "Qwen/Qwen3.8-27B", "Qwen/Qwen3.8-2.4T-A95B",
         "google/gemma-3-27b-it", "google/gemma-4-12B-it", "meta-llama/Llama-3.3-70B-Instruct", "meta-llama/Llama-4-Scout-17B-16E-Instruct",
         "zai-org/GLM-5.3", "zai-org/GLM-5.3-Flash", "moonshotai/Kimi-K3.1-Pro", "mistralai/Mistral-Large-Instruct-2411",
         "mistralai/Mistral-Large-4-1T-A52B", "CohereLabs/c4ai-command-a-03-2025", "bigscience/bloom"]
rows = []
for r in REPOS:
    try:
        rows.append(read_license(r))
    except Exception as e:                                     # a repository may be renamed or removed later
        rows.append({"repo": r, "license": f"<{type(e).__name__}>", "name": None, "gated": False, "category": "unknown"})
assert len(survey(REPOS[:2])) == 2 and survey(REPOS[:2])[1]["category"] == "permissive"

print("m62_hub: all tests passed")
print(f"  {'repository':44s} {'license':26s} {'gated':6s} category")
for x in rows:
    lic = str(x["license"]) + (f" ({x['name']})" if x["name"] else "")
    print(f"  {x['repo']:44s} {lic:26s} {'yes' if x['gated'] else 'no':6s} {x['category']}")

<details><summary><b>提示 1 / Hint 1</b></summary>

先把标识符和名称都转成小写，`name` 可能是 `None`，可以写成 `(name or '').lower()`。

Convert both identifier and name to lower case. `name` may be `None`, so write `(name or '').lower()`.

</details>

<details><summary><b>提示 2 / Hint 2</b></summary>

`info.card_data` 可能是 `None`。取字段用 `card.get('license')`，缺失时得到 `None`。

`info.card_data` may be `None`. Read fields with `card.get('license')`, which gives `None` when missing.

</details>

<details><summary><b>提示 3 / Hint 3</b></summary>

`info.gated` 的取值是 `False`、`'auto'` 或 `'manual'`，用 `bool(info.gated)` 转成布尔值。

`info.gated` is `False`, `'auto'` or `'manual'`, converted to a boolean with `bool(info.gated)`.

</details>

<details><summary><b>参考答案 / Reference solution</b></summary>

```python
from huggingface_hub import HfApi

PERMISSIVE = {"apache-2.0", "mit", "bsd-2-clause", "bsd-3-clause", "cc0-1.0", "cc-by-4.0", "isc", "unlicense"}
COPYLEFT = {"gpl-2.0", "gpl-3.0", "agpl-3.0", "lgpl-3.0", "mpl-2.0", "cc-by-sa-4.0"}


def classify_license(identifier, name=None):
    if not identifier or identifier == "unknown":
        return "unknown"
    ident = identifier.lower()
    label = (name or "").lower()
    if ident in PERMISSIVE:
        return "permissive"
    if ident in COPYLEFT:
        return "copyleft"
    if "-nc" in ident or "noncommercial" in ident or "research" in ident or "research" in label or label == "mrl":
        return "non-commercial"
    if "rail" in ident:
        return "use-restricted"
    return "custom"                                         # family licenses (llama*, gemma) and everything declared as "other"


def read_license(repo_id, api=None):
    info = (api or HfApi()).model_info(repo_id)
    card = info.card_data.to_dict() if info.card_data else {}
    identifier, name = card.get("license"), card.get("license_name")
    return {"repo": repo_id, "license": identifier, "name": name, "gated": bool(info.gated),
            "category": classify_license(identifier, name)}


def survey(repo_ids):
    api = HfApi()
    return [read_license(r, api) for r in repo_ids]
```

</details>

## 2. 许可证情景判断 / License scenarios

根据本模块正文中引用的许可证条款，对下面每个情景给出初步判断并说明依据。这些判断用于练习阅读条款，不构成法律意见。
(a) 一家初创公司想把 `Qwen/Qwen3.8-27B`（Apache 2.0）微调后作为付费产品的一部分，并把微调后的权重提供给客户。
(b) 一名博士生用 `mistralai/Mistral-Large-Instruct-2411`（Mistral 研究许可证）做论文实验。毕业后她加入一家公司，想用同一个模型做内部的概念验证。
(c) 一个研究组在 `meta-llama/Llama-3.3-70B-Instruct` 上微调出一个地理问答模型，准备以 GeoQA-70B 的名字发布到 Hub。
(d) 一名研究者在 `google/gemma-3-27b-it` 上完成了全部实验，同事建议换用 Gemma 4 以便更自由地发布衍生模型。这个建议有依据吗？
(e) 一名研究者想研究模型在回答地理问题时内部各层的激活，手头有某个闭源模型的 API 额度和一块 24 GB 的显卡。她应当怎样安排？

Using the license terms quoted in the text of this module, give a first judgement for each scenario below and state its basis. These judgements serve as practice in reading terms and are not legal advice.
(a) A start-up wants to fine-tune `Qwen/Qwen3.8-27B` (Apache 2.0), use it as part of a paid product and provide the fine-tuned weights to customers.
(b) A doctoral student runs thesis experiments with `mistralai/Mistral-Large-Instruct-2411` (Mistral AI Research License). After graduating she joins a company and wants to use the same model for an internal proof of concept.
(c) A research group fine-tunes a geographic question-answering model from `meta-llama/Llama-3.3-70B-Instruct` and plans to publish it on the Hub under the name GeoQA-70B.
(d) A researcher has completed all experiments on `google/gemma-3-27b-it`, and a colleague suggests switching to Gemma 4 in order to release derived models more freely. Is there a basis for this suggestion?
(e) A researcher wants to study the activations in the layers of a model while it answers geographic questions and has API credit for a closed model and a 24 GB GPU. How should she proceed?

<details><summary><b>提示 1 / Hint 1</b></summary>

(b) 注意研究许可证对“研究用途”的定义排除了哪些活动。

(b) Note which activities the definition of research purposes in the research license excludes.

</details>

<details><summary><b>提示 2 / Hint 2</b></summary>

(c) 查看 Llama 3.3 协议中关于衍生模型命名的要求。

(c) Look at the requirement on naming derived models in the Llama 3.3 agreement.

</details>

<details><summary><b>参考答案 / Reference solution</b></summary>

(a) 初步判断为可以。Apache 2.0 不限制商用，允许修改和再分发。向客户提供权重属于再分发，需要附上许可证副本、保留原有的声明、标明所做的修改。该许可证不授予商标权，产品宣传中使用 Qwen 的名称和标识需要另行确认。

(b) 论文实验属于学术研究，符合该许可证允许的研究用途。公司内部的概念验证则不在许可范围之内，该许可证明确把员工在日常工作中的使用和以创收为目的的活动（包括测试和概念验证）排除在研究用途之外。公司需要向 Mistral 另行取得商用许可，或者改用许可证允许商用的模型。

(c) 这个名字不符合要求。Llama 3.3 的协议规定，使用该模型训练或微调并对外发布的模型，名称必须以 Llama 开头。发布时还要附上协议副本，在相关页面的显著位置标注 Built with Llama，并遵守可接受使用政策。可以改为以 Llama 开头的名称，例如 Llama-3.3-GeoQA-70B。

(d) 有依据。根据 Hub 的元数据和 Google 的许可页面，Gemma 4 采用 Apache 2.0，而 Gemma 3 采用自定义的 Gemma 使用条款，后者要求把使用限制传递给下游并保留了发布者限制违规使用的权利。不过更换模型意味着重做实验，是否值得取决于发布衍生模型的需要有多强。已经完成的 Gemma 3 实验在其条款下写进论文通常没有障碍。

(e) 读取内部激活需要持有权重，闭源模型的 API 做不到这一点。可以选用许可证允许研究用途、并且以某种精度能放进 24 GB 显存的开放权重模型做机制分析，例如一个 270 亿参数的模型以 4 位量化后约 14 GB。需要注意的是量化会改变激活值，论文中应当说明所用的精度。闭源模型可以用于行为层面的对照实验，即比较输入和输出，不涉及内部状态。

---

(a) A first judgement is that this is allowed. Apache 2.0 does not restrict commercial use and permits modification and redistribution. Providing weights to customers is redistribution, which requires including a copy of the license, keeping existing notices and marking the changes made. The license grants no trademark rights, so using the name and marks of Qwen in product promotion needs separate confirmation.

(b) Thesis experiments are academic research and fall under the research purposes the license allows. An internal proof of concept at a company is outside the license, which explicitly excludes use by employees in their daily tasks and revenue-generating activities (including testing and proofs of concept) from research purposes. The company has to obtain a separate commercial license from Mistral or switch to a model whose license allows commercial use.

(c) This name does not meet the requirement. The Llama 3.3 agreement provides that a model trained or fine-tuned with the model and then distributed must carry a name beginning with Llama. The release must also include a copy of the agreement, prominently display Built with Llama on a related page and comply with the acceptable use policy. The name can be changed to one beginning with Llama, such as Llama-3.3-GeoQA-70B.

(d) There is a basis. According to the Hub metadata and the license page of Google, Gemma 4 is under Apache 2.0, whereas Gemma 3 is under the custom Gemma Terms of Use, which require passing use restrictions downstream and reserve the right of the publisher to restrict violating usage. Switching models means redoing the experiments, however, and whether that is worthwhile depends on how strongly derived models need to be released. Reporting the completed Gemma 3 experiments in a paper under its terms usually meets no obstacle.

(e) Reading internal activations requires the weights, which the API of a closed model cannot provide. Mechanism analysis can be done with an open-weight model whose license allows research use and which fits into 24 GB at some precision, for example a model with 27 billion parameters at 4 bits, about 14 GB. Note that quantization changes the activations, and the paper should state the precision used. The closed model can serve for behavioural comparison experiments, that is, comparing inputs and outputs without reference to internal states.

</details>